# Data Collection & Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
#Load data file
orders = pd.read_excel('Dataset.xlsx')
orders

# Data Preparation & Cleaning

In [ ]:
#Column info (Check type + NA value)
orders.info()

In [ ]:
#Load data from "Return" sheet
returned = pd.read_excel('Dataset.xlsx', sheet_name='Return')
returned

In [ ]:
#Remove returned orders
orders2 = orders.merge(returned, how="left", on="Order ID").reset_index()
success_orders = orders2[orders2.Returned.isna()]
success_orders

# Profitability Analysis

In [ ]:
# Calculate Profit

success_orders = success_orders.copy()

success_orders['Profit'] = (
    success_orders['Sales']
    - success_orders['Quantity'] * success_orders['Unit Cost']
)

success_orders

In [ ]:
#Calculate RFM
#Recency
max_orderdate = success_orders.groupby('Customer ID')['Order Date'].max().reset_index()
max_orderdate.columns = ['Customer ID', 'Max Order Date']
max_orderdate

In [ ]:
import datetime
current_date = datetime.datetime(2017,12,31)
max_orderdate ['Recency']  = (current_date - max_orderdate['Max Order Date']).dt.days
max_orderdate

In [ ]:
#Frequency
order_fre = success_orders.groupby('Customer ID')['Order ID'].nunique().reset_index()
order_fre.columns = ['Order ID', 'Frequency']
order_fre

In [ ]:
#Monetary
rfm_cal = success_orders.groupby(['Customer ID']).agg(
    {'Order Date': lambda x: (current_date - x.max()).days,
    'Order ID': 'nunique',
    "Sales":'sum',
    'Quantity':'sum',
    'Profit':'sum'}
).reset_index()
rfm_cal

In [ ]:
#Rename columns
rfm_cal.rename(columns={'Order Date':'Recency','Order ID':'Frequency','Sales':'Monetary'}, inplace=True)
rfm_cal

In [ ]:
#Calculate quintiles
rfm_cal['Rec_score'] = pd.qcut(rfm_cal['Recency'],5,[5,4,3,2,1])
rfm_cal['Fre_score'] = pd.qcut(rfm_cal['Frequency'],5,[1,2,3,4,5])
rfm_cal['Mon_score'] = pd.qcut(rfm_cal['Monetary'],5,[1,2,3,4,5])
rfm_cal

In [ ]:
#RFM score segmentation
rfm_cal['RFM_score'] = rfm_cal['Rec_score'].astype(str) + rfm_cal['Fre_score'].astype(str) + rfm_cal['Mon_score'].astype(str)
rfm_cal['RFM_score'] = pd.to_numeric(rfm_cal['RFM_score'])
rfm_cal

In [ ]:
#Load segment data
seg = pd.read_excel('Dataset.xlsx', sheet_name='Segmentation')
seg

In [ ]:
#Add segment data to list
seg['RFM Score'] = seg['RFM Score'].str.split(",")
seg

In [ ]:
#Add refer table
refer_table = seg.set_index(["Segment"])['RFM Score'].apply(pd.Series).stack().reset_index().drop(columns='level_1').rename(columns={0: 'RFM_score'})
refer_table['RFM_score'] = pd.to_numeric(refer_table['RFM_score'])
refer_table

In [ ]:
#Merge scores to appropriate segments
rfm_cal = rfm_cal.merge(refer_table, how="left", on="RFM_score").reset_index().drop(columns='index')
rfm_cal

# Customer Segmentation
## Segment Analysis

In [ ]:
#Draw bar chart to show number of customers per segment
category_order = ['Champions', 
'Loyal',
'Potential Loyalist',
'New Customers',
'Promising',
'Need Attention',
'About To Sleep',
'At Risk',
'Cannot Lose Them',
'Hibernating customers',
'Lost customers']

sns.set_style("whitegrid")
g = sns.catplot(y="Segment", data=rfm_cal, orient='h', kind="count", order=category_order)
g.fig.suptitle("Number of Customers per Segment", y=1.03)
g.set(xlabel="Total Customers")
plt.show()

In [ ]:
# Revenue and Profit per Segment
#Calculate rev_per_seg
rev_per_seg = rfm_cal.groupby(['Segment']).agg(
    {'Monetary':'sum',
     'Profit':'sum'}).reset_index()

#Draw chart
x = rev_per_seg['Segment']
y1 = rev_per_seg['Monetary']
y2 = rev_per_seg['Profit']


col1 = 'steelblue'
col2 = 'red'

fig,ax = plt.subplots()

ax.bar(x, y1, color=col1)


ax.set_xlabel('Segment', fontsize=14)
plt.xticks(rotation=75)

ax.set_ylabel('Revenue', color=col1, fontsize=14)


ax2 = ax.twinx()


ax2.plot(x, y2, color=col2)

ax2.set_ylabel('Profit', color=col2, fontsize=14)
plt.grid(False)
plt.title("Revenue and Profit by Segments")
plt.show()

### Customer Insights & Segment Analysis

In [ ]:
#Load data from "Product" sheet
product = pd.read_excel('Dataset.xlsx', sheet_name='Product')
product

In [ ]:
#Merge "product" vs. "potential" vs. "success_orders"
order3 = success_orders.merge(product, how="left", on="Product ID").reset_index(drop=True).drop(columns='index')
order4 = order3.merge(rfm_cal, how="left", on="Customer ID").reset_index().drop(columns='index')
order4

In [ ]:
#Filter Potential Loyal list customer
potential = order4[(order4.Segment == 'Potential Loyalist')]
potential

In [ ]:
#Calculate Revenue per Channel, Category and Ship Mode
chan = order4.groupby(['Channel']).agg({'Sales':'sum'}).reset_index()

cat = order4.groupby(['Category']).agg({'Sales':'sum'}).reset_index()

ship = order4.groupby(['Ship Mode']).agg({'Sales':'sum'}).reset_index()


In [ ]:
#Revenue per Channel, Category and Ship Mode
import plotly.express as px
import plotly

fig = px.pie(chan, values= 'Sales', names= 'Channel', title = 'Revenue by Channel')
fig.show()

fig = px.pie(cat, values= 'Sales', names= 'Category', title = 'Revenue by Category')
fig.show()

fig = px.pie(ship, values= 'Sales', names= 'Ship Mode', title = 'Revenue by Ship Mode')
fig.show()


In [ ]:
#Extract Month and Year
time = order4.loc[:,['Order Date', 'Sales']]
time['Year'] = time['Order Date'].dt.year 
time['Month'] = time['Order Date'].dt.month 
time

In [ ]:
#Change of Sales overtime (Potential Royalist only)
sale_line = time.groupby(['Year','Month'], as_index=False)['Sales'].sum().round(2)

fig = px.line (sale_line, x='Month', y='Sales', color='Year', markers=True, title="Sales Overtime")
fig.show()

# Business Recommendations

1. Reward Champions with loyalty programs.
2. Re-engage At Risk customers through targeted campaigns.
3. Convert Potential Loyalists into repeat customers.
4. Focus marketing efforts on high-value customer segments.
5. Improve retention strategies to reduce customer churn.